# Hafta 8: Birliktelik Kuralı Madenciliği

Bu notebook'ta aşağıdaki konular ele alınmaktadır:
1. Support, Confidence, Lift metriklerinin manuel hesaplanması
2. Transaction verisinin hazırlanması (one-hot encoding)
3. Apriori ile sık itemset bulma
4. Association Rules ve metrik analizi
5. Conviction, Leverage ve Kulpa metriği
6. Eşik (min_support/min_confidence) duyarlılık analizi
7. FP-Growth ile performans karşılaştırması
8. Kural ağının (network) görselleştirilmesi
9. Sonuç ve iş uygulaması yorumları

## 1. Gerekli Kütüphanelerin Yüklenmesi

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

# mlxtend kontrol
try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
    HAS_MLXTEND = True
except ImportError:
    HAS_MLXTEND = False

# networkx kontrol
try:
    import networkx as nx
    HAS_NETWORKX = True
except ImportError:
    HAS_NETWORKX = False

print('mlxtend durum:', 'Hazır' if HAS_MLXTEND else 'Eksik (pip install mlxtend)')
print('networkx durum:', 'Hazır' if HAS_NETWORKX else 'Eksik (pip install networkx)')

## 2. Temel Kavramlar: Metriklerin Manuel Hesabı

### Formüller
$$Support(X) = P(X)$$
$$Confidence(X \rightarrow Y) = \frac{Support(X \cup Y)}{Support(X)}$$
$$Lift(X \rightarrow Y) = \frac{Support(X \cup Y)}{Support(X) \times Support(Y)}$$

Ek metrikler:
$$Conviction(X \rightarrow Y) = \frac{1 - Support(Y)}{1 - Confidence(X \rightarrow Y)}$$
$$Leverage(X \rightarrow Y) = Support(X \cup Y) - Support(X)Support(Y)$$
$$Kulpa(X \rightarrow Y) = \frac{Support(X \cup Y) - Support(X)Support(Y)}{Support(X)(1 - Support(X))}$$

In [ ]:
# Kucuk bir oyuncak veri ile manuel hesap
toy_transactions = [
    {'Ekmek', 'Sut'},
    {'Ekmek', 'Sut', 'Peynir'},
    {'Sut', 'Peynir'},
    {'Ekmek', 'Sut', 'Yumurta'},
    {'Ekmek', 'Yumurta'}
]

N = len(toy_transactions)
def support(itemset):
    return sum(itemset.issubset(t) for t in toy_transactions) / N

X = {'Ekmek'}
Y = {'Sut'}
sX = support(X)
sY = support(Y)
sXY = support(X.union(Y))
conf = sXY / sX if sX > 0 else 0
lift = sXY / (sX * sY) if sX > 0 and sY > 0 else 0
conv = (1 - sY) / (1 - conf) if conf < 1 else np.inf
lev = sXY - sX * sY
kulpa = lev / (sX * (1 - sX)) if 0 < sX < 1 else np.nan

print('=' * 55)
print('Manuel Metrik Hesabi: Ekmek -> Sut')
print('=' * 55)
print(f'Support(Ekmek)        = {sX:.3f}')
print(f'Support(Sut)          = {sY:.3f}')
print(f'Support(Ekmek,Sut)    = {sXY:.3f}')
print(f'Confidence            = {conf:.3f}')
print(f'Lift                  = {lift:.3f}')
print(f'Conviction            = {conv:.3f}')
print(f'Leverage              = {lev:.3f}')
print(f'Kulpa                 = {kulpa:.3f}')

## 3. Veri Seti Yükleme

Kullanılan dosya: `Veri_Setleri_Genel/transactions.csv`

Beklenen format:
- `TransactionID`: işlem numarası
- `Items`: virgül ile ayrılmış ürün listesi

In [ ]:
import os

DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                         '..', '..', 'Veri_Setleri_Genel', 'transactions.csv')
df_raw = pd.read_csv(DATA_PATH)

print(f'Veri seti boyutu: {df_raw.shape[0]} satir, {df_raw.shape[1]} sutun')
print('\nIlk 10 satir:')
display(df_raw.head(10))

In [ ]:
# Islem bazli sepet listesi olustur
transactions = (
    df_raw['Items']
    .fillna('')
    .apply(lambda x: [item.strip() for item in x.split(',') if item.strip() != ''])
    .tolist()
)

print(f'Toplam islem sayisi: {len(transactions)}')
all_items = sorted(set(item for t in transactions for item in t))
print(f'Toplam benzersiz urun: {len(all_items)}')
print('Urunler:', all_items)

basket_sizes = [len(t) for t in transactions]
print(f'
Ortalama sepet boyutu: {np.mean(basket_sizes):.2f}')
print(f'Min / Max sepet boyutu: {np.min(basket_sizes)} / {np.max(basket_sizes)}')

In [ ]:
# Sepet boyutu dagilimi
plt.figure(figsize=(8, 4))
sns.countplot(x=basket_sizes, color='steelblue')
plt.title('Sepet Boyutu Dagilimi', fontsize=13, fontweight='bold')
plt.xlabel('Sepetteki urun sayisi')
plt.ylabel('Islem sayisi')
plt.tight_layout()
plt.show()

## 4. One-Hot Encoding (TransactionEncoder)

Apriori ve FP-Growth algoritmaları için veri, ürün bazlı binary matrise çevrilir.

In [ ]:
if not HAS_MLXTEND:
    raise ImportError('Bu notebook icin mlxtend gerekli. Lutfen: pip install mlxtend')

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

print('One-hot matris boyutu:', basket_df.shape)
display(basket_df.head())

# Urun destekleri (tek item support)
item_support = basket_df.mean().sort_values(ascending=False)
display(item_support.to_frame('support'))

In [ ]:
plt.figure(figsize=(8, 4))
item_support.sort_values(ascending=True).plot(kind='barh', color='teal')
plt.title('Tekil Urun Support Degerleri', fontsize=13, fontweight='bold')
plt.xlabel('Support')
plt.tight_layout()
plt.show()

## 5. Apriori ile Sık Itemset Bulma

Apriori ilkesi: Eğer bir itemset sık değilse, onun tüm supersetleri de sık değildir.

In [ ]:
# Farkli min_support esiklerinde itemset sayisi
support_values = [0.10, 0.20, 0.30, 0.40, 0.50]
itemset_counts = []

for s in support_values:
    fi = apriori(basket_df, min_support=s, use_colnames=True)
    itemset_counts.append(len(fi))

plt.figure(figsize=(8, 4))
plt.plot(support_values, itemset_counts, 'o-', color='darkorange', linewidth=2)
for x, y in zip(support_values, itemset_counts):
    plt.text(x, y + 0.1, str(y), ha='center')
plt.title('Min Support Esigine Gore Itemset Sayisi', fontsize=13, fontweight='bold')
plt.xlabel('min_support')
plt.ylabel('Sik itemset sayisi')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Ana esik
MIN_SUPPORT = 0.20
frequent_itemsets = apriori(basket_df, min_support=MIN_SUPPORT, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)

frequent_itemsets_sorted = frequent_itemsets.sort_values(['length', 'support'], ascending=[True, False])
print(f'min_support={MIN_SUPPORT} ile bulunan itemset sayisi: {len(frequent_itemsets_sorted)}')
display(frequent_itemsets_sorted.reset_index(drop=True))

In [ ]:
# Itemset uzunluguna gore dagilim
length_counts = frequent_itemsets['length'].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(length_counts.index.astype(str), length_counts.values, color='slateblue')
plt.title('Sik Itemset Uzunluk Dagilimi', fontsize=13, fontweight='bold')
plt.xlabel('Itemset uzunlugu')
plt.ylabel('Sayi')
for i, v in zip(length_counts.index, length_counts.values):
    plt.text(i - 1, v + 0.1, str(v))
plt.tight_layout()
plt.show()

## 6. Association Rules Çıkarma

Kurallar `association_rules` fonksiyonu ile confidence, lift gibi metriklere göre üretilir.

In [ ]:
MIN_CONF = 0.60
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=MIN_CONF)

if len(rules) == 0:
    print('Bu esiklerde kural bulunamadi. min_support/min_confidence degerlerini dusurun.')
else:
    # Kulpa metrigini manuel ekleyelim
    rules['kulpa'] = (
        rules['support'] - rules['antecedent support'] * rules['consequent support']
    ) / (rules['antecedent support'] * (1 - rules['antecedent support']))

    # Okunabilirlik
    rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
    rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(list(x))))

    cols = [
        'antecedents_str', 'consequents_str',
        'support', 'confidence', 'lift', 'conviction', 'leverage', 'kulpa'
    ]
    rules_view = rules[cols].sort_values('lift', ascending=False).reset_index(drop=True)

    print(f'min_confidence={MIN_CONF} ile bulunan kural sayisi: {len(rules_view)}')
    display(rules_view.round(4))

In [ ]:
if len(rules) > 0:
    # En iyi 10 kural
    top_rules = rules.sort_values(['lift', 'confidence'], ascending=[False, False]).head(10).copy()
    top_rules['rule'] = top_rules.apply(
        lambda r: f"{', '.join(sorted(r['antecedents']))} -> {', '.join(sorted(r['consequents']))}",
        axis=1
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_rules, x='lift', y='rule', color='indianred')
    plt.title('En Yuksek Lift Degerine Sahip 10 Kural', fontsize=13, fontweight='bold')
    plt.xlabel('Lift')
    plt.ylabel('Kural')
    plt.tight_layout()
    plt.show()

    # Scatter: support vs confidence, renk=lift
    plt.figure(figsize=(7, 5))
    sc = plt.scatter(rules['support'], rules['confidence'], c=rules['lift'],
                     cmap='viridis', s=90, alpha=0.8, edgecolors='k')
    plt.colorbar(sc, label='Lift')
    plt.title('Kurallar: Support vs Confidence', fontsize=13, fontweight='bold')
    plt.xlabel('Support')
    plt.ylabel('Confidence')
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

## 7. Eşik Duyarlılık Analizi

Farklı `min_support` ve `min_confidence` değerlerinde kaç kural üretildiğini inceleyelim.

In [ ]:
support_grid = [0.10, 0.15, 0.20, 0.25, 0.30]
conf_grid = [0.50, 0.60, 0.70, 0.80]

heat_data = pd.DataFrame(index=support_grid, columns=conf_grid, dtype=float)

for s in support_grid:
    fi = apriori(basket_df, min_support=s, use_colnames=True)
    for c in conf_grid:
        r = association_rules(fi, metric='confidence', min_threshold=c) if len(fi) > 0 else pd.DataFrame()
        heat_data.loc[s, c] = len(r)

plt.figure(figsize=(8, 5))
sns.heatmap(heat_data, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('Esik Duyarlilik: Kural Sayisi', fontsize=13, fontweight='bold')
plt.xlabel('min_confidence')
plt.ylabel('min_support')
plt.tight_layout()
plt.show()

display(heat_data)

## 8. FP-Growth ile Karşılaştırma

FP-Growth, Apriori'ye göre aday itemset üretmeden çalışır ve genelde daha hızlıdır.

In [ ]:
SUPPORT_FOR_COMPARE = 0.20

# Apriori sure
t0 = time.time()
fi_apriori = apriori(basket_df, min_support=SUPPORT_FOR_COMPARE, use_colnames=True)
apriori_time = time.time() - t0

# FP-Growth sure
t0 = time.time()
fi_fp = fpgrowth(basket_df, min_support=SUPPORT_FOR_COMPARE, use_colnames=True)
fp_time = time.time() - t0

comparison_df = pd.DataFrame({
    'Algoritma': ['Apriori', 'FP-Growth'],
    'Sure (sn)': [apriori_time, fp_time],
    'Itemset Sayisi': [len(fi_apriori), len(fi_fp)]
})

display(comparison_df.round(6))

plt.figure(figsize=(7, 4))
sns.barplot(data=comparison_df, x='Algoritma', y='Sure (sn)', palette=['steelblue', 'seagreen'])
plt.title('Apriori vs FP-Growth Sure Karsilastirmasi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Kural Ağı Görselleştirmesi

Lift değeri yüksek kurallar yönlü grafik olarak gösterilir.

In [ ]:
if len(rules) == 0:
    print('Gorsellestirme icin yeterli kural yok.')
elif not HAS_NETWORKX:
    print('networkx kurulu degil. Kurulum: pip install networkx')
else:
    # Cok karmasik olmamasi icin ilk 12 kural
    top_for_graph = rules.sort_values('lift', ascending=False).head(12).copy()

    G = nx.DiGraph()
    for _, row in top_for_graph.iterrows():
        a = ', '.join(sorted(list(row['antecedents'])))
        c = ', '.join(sorted(list(row['consequents'])))
        G.add_edge(a, c, weight=row['lift'], confidence=row['confidence'])

    plt.figure(figsize=(10, 7))
    pos = nx.spring_layout(G, k=1.2, seed=42)

    # Dugumler
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1800, edgecolors='black')

    # Kenarlar (lift'e gore kalinlik)
    edges = G.edges(data=True)
    widths = [1 + 2 * e[2]['weight'] for e in edges]
    nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=18, width=widths, alpha=0.7)

    # Etiketler
    nx.draw_networkx_labels(G, pos, font_size=9)

    # Kenar etiketi: lift
    edge_labels = {(u, v): f"L={d['weight']:.2f}" for u, v, d in edges}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

    plt.title('Association Rules Network (Top Lift Kurallari)', fontsize=13, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 10. İş Yorumları: Top Kurallar

Yüksek lift ve yeterli confidence değerine sahip kurallar, çapraz satış ve kampanya kararlarında kullanılabilir.

In [ ]:
if len(rules) > 0:
    business_rules = (
        rules
        .query('lift > 1.0 and confidence >= 0.6')
        .sort_values(['lift', 'confidence'], ascending=[False, False])
        .head(10)
        .copy()
    )

    if len(business_rules) == 0:
        print('Belirlenen filtrede (lift>1 ve conf>=0.6) kural bulunamadi.')
    else:
        business_rules['Kural'] = business_rules.apply(
            lambda r: f"{', '.join(sorted(r['antecedents']))} -> {', '.join(sorted(r['consequents']))}",
            axis=1
        )
        out_cols = ['Kural', 'support', 'confidence', 'lift', 'conviction', 'leverage']
        display(business_rules[out_cols].round(4).reset_index(drop=True))

        print('Oneri ornegi:')
        best = business_rules.iloc[0]
        a = ', '.join(sorted(best['antecedents']))
        c = ', '.join(sorted(best['consequents']))
        print(f'- {a} alan musterilere {c} icin kampanya/oneri gosterilebilir.')
        print(f'- Bu kuralin confidence degeri %{best[confidence]*100:.1f}, lift degeri {best[lift]:.2f}.')

## 11. Sonuç

Bu uygulamada:
- Apriori ile sık itemset ve kurallar çıkarıldı
- Support, Confidence, Lift dışında Conviction, Leverage ve Kulpa incelendi
- Eşik değerlerinin kural sayısına etkisi analiz edildi
- FP-Growth ile hız karşılaştırması yapıldı
- Kurallar ağ yapısı ile görselleştirildi

Pratikte modelleme adımı genelde şu şekilde ilerler:
1. İş hedefi belirlenir (öneri, çapraz satış, raf yerleşimi)
2. Uygun min_support ve min_confidence aralığı aranır
3. Lift ve leverage ile sahte kurallar elenir
4. Domain bilgisi ile anlamlı kurallar seçilir
5. A/B test ile iş etkisi doğrulanır